In [1]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


# Carga Inicial de Datos

[Sentiment140](https://huggingface.co/datasets/stanfordnlp/sentiment140) es un conjunto de datos de gran escala para la clasificación del sentimiento en Twitter, que contiene 1,6 millones de tuits etiquetados automáticamente mediante emoticonos como forma de supervisión distante (*distant supervision*).

Los tuits que contienen emoticonos positivos (p. ej., :), :-)) se etiquetan como positivos (`sentiment=4`), mientras que los que contienen emoticonos negativos (p. ej., :(, :-() se etiquetan como negativos (`sentiment=0`). Este conjunto de datos se utiliza ampliamente como referencia para la clasificación de sentimientos binaria y multiclase en la investigación sobre procesamiento del lenguaje natural (PLN).



In [2]:
import io, zipfile, urllib.request
import re, html
import pandas as pd

_URL = "http://help.sentiment140.com/home"
_DATA_URL = "https://cs.stanford.edu/people/alecmgo/trainingandtestdata.zip"

_TRAIN_FILE_NAME = "training.1600000.processed.noemoticon.csv"

# Descargamos el zip con el Dataset
with urllib.request.urlopen(_DATA_URL) as r:
    zf = zipfile.ZipFile(io.BytesIO(r.read()))


# Pre-processing

Partimos de texto generalmente muy informal con abreviaciones, faltas de ortografia, #, @ de usuarios y un monton de palabras y abreviaciones polisemicas.

Por lo que si queremos crear un clasificador debemos de limpiar los datos lo maximo posible. Esto es lo que hemos hecho:

- **Unificamos el formato del texto**: lo pasamos a minúsculas, decodificamos las entidades HTML y normalizamos los apóstrofos.

- **Sustituimos los elementos variables por tokens genéricos**: URLs, menciones, hashtags, números, caras y corazones pasan a ser `<url>`, `<user>`, `<hashtag>`, `<num>`, `<happyface>`, `<sadface>` y `<heart>`. Lo que nos interesa es que aparecen, no su contenido concreto.

- **Expandimos contracciones y abreviaturas** (`can't` → `can not`, `dont` → `do not`, `btw` → `by the way`). Así las negaciones quedan como palabras sueltas y el modelo puede aprenderlas.

- **Reducimos la intensidad sin perderla**: acortamos las letras y exclamaciones repetidas (`soooo` → `soo`, `!!!!` → `!!`) en lugar de eliminarlas.

- **Eliminamos el resto de símbolos** y dividimos el texto en tokens, conservando `!` y `?` porque aportan carga emocional.



> Dicho esto hay problemas que **no vamos a poder corregir**, como la **polisemia** de abreviaciones y palabras, **faltas de ortografía**.



In [3]:

cols = ["target", "id", "date", "flag", "user", "text"]
df = pd.read_csv(zf.open(_TRAIN_FILE_NAME),
                 encoding="latin-1", header=None, names=cols)

print(df.head())

# Nos quedamos solo con target y text. Convertimos el sentimiento
# que en el dataset es 0 = negativo, 4 = positivo a una c.binaria
df = df[["target", "text"]]
df["label"] = (df["target"] == 4).astype(int)   # 0 = negativo, 1 = positivo


   target          id                          date      flag  \
0       0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY   
1       0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY   
2       0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY   
3       0  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   
4       0  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   

              user                                               text  
0  _TheSpecialOne_  @switchfoot http://twitpic.com/2y1zl - Awww, t...  
1    scotthamilton  is upset that he can't update his Facebook by ...  
2         mattycus  @Kenichan I dived many times for the ball. Man...  
3          ElleCTF    my whole body feels itchy and like its on fire   
4           Karoli  @nationwideclass no, it's not behaving at all....  


In [4]:
# Contracciones y abreviaciones tipicamente usadas en ingles
CONTRACTIONS = [
    (r"\bwon't\b", "will not"), (r"\bcan't\b", "can not"), (r"\bain't\b", "is not"),
    (r"\b(it|that|what|there|he|she|who|here|where)'s\b", r"\1 is"),
    (r"n't\b", " not"), (r"'re\b", " are"), (r"'m\b", " am"), (r"'ll\b", " will"),
    (r"'ve\b", " have"), (r"'d\b", " would"), (r"'s\b", ""), (r"\bbtw\b", "by the way"),
    (r"\btbh\b", "to be honest"), (r"\bfr\b", "for real")
]
CONTRACTIONS = [(re.compile(p), r) for p, r in CONTRACTIONS]

# Corregimos apostrofes que puedan faltar, hoy en dia el corrector lo suele evitar
NO_APOSTROPHE = {
    "dont": "do not", "doesnt": "does not", "didnt": "did not", "isnt": "is not",
    "wasnt": "was not", "arent": "are not", "werent": "were not", "havent": "have not",
    "hasnt": "has not", "couldnt": "could not", "wouldnt": "would not",
    "shouldnt": "should not", "cant": "can not", "wont": "will not",
    "im": "i am", "ive": "i have", "youre": "you are", "thats": "that is",
}
NO_APOSTROPHE_RE = re.compile(r"\b(" + "|".join(NO_APOSTROPHE) + r")\b")

# Expresiones para detectar urls, usuarios, hastaghs, numeros, enfasis, :D, :(
URL_RE     = re.compile(r"https?://\S+|www\.\S+")
USER_RE    = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#(\w+)")
NUM_RE     = re.compile(r"-?\b(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?\b")
LONG_RE  = re.compile(r"([a-z])\1{2,}")            # soooo -> soo
EXCLAM_RE  = re.compile(r"([!?])\1+")              # !!!!  -> !!
REST_RE   = re.compile(r"[^a-z<>!?\s]")            # solo letras, ! y ?
HAPPY_FACE = re.compile(r"[:;=]-?\)+|[:;=]-?d+\b") # ;) :) :d :))))
SAD_FACE   = re.compile(r"[:;=]-?\(+")             # :( :((( ;-(
TOKEN_RE   = re.compile(r"<url>|<user>|<hashtag>|<num>|<heart>|<happyface>|<sadface>|[a-z]+|[!?]{1,2}")


def preprocess(texto) :
  t = html.unescape(texto).lower()                 # convertimos a minúsculas
  t = t.replace("\u2019", "'").replace("`", "'")   # quitamos apóstrofos raros
  t = URL_RE.sub(" <url> ", t)                     # marcamos urls
  t = USER_RE.sub(" <user> ", t)                   # marcamos usuarios
  t = HAPPY_FACE.sub(" <happyface> ", t)           # marcamos ;)
  t = SAD_FACE.sub(" <sadface> ", t)               # marcamos :(
  t = HASHTAG_RE.sub(r" <hashtag> \1 ", t)         # marcamos hastags
  t = t.replace("<3", " <heart> ")                 # marcamos corazones

  for patron, rep in CONTRACTIONS:
      t = patron.sub(rep, t)

  t = NO_APOSTROPHE_RE.sub(lambda m: NO_APOSTROPHE[m.group(1)], t)

  t = NUM_RE.sub(" <num> ", t)           # marcamos numeros
  t = LONG_RE.sub(r"\1\1", t)            # acortamos enfasis palabra
  t = EXCLAM_RE.sub(r" \1\1 ", t)        # acortamos enfasis exclamaciones
  t = REST_RE.sub(" ", t)                # eliminamos caracteres no permitidos

  return TOKEN_RE.findall(t)

df["tokens"] = df["text"].apply(preprocess)

In [5]:
print(df.head())

   target                                               text  label  \
0       0  @switchfoot http://twitpic.com/2y1zl - Awww, t...      0   
1       0  is upset that he can't update his Facebook by ...      0   
2       0  @Kenichan I dived many times for the ball. Man...      0   
3       0    my whole body feels itchy and like its on fire       0   
4       0  @nationwideclass no, it's not behaving at all....      0   

                                              tokens  
0  [<user>, <url>, aww, that, is, a, bummer, you,...  
1  [is, upset, that, he, can, not, update, his, f...  
2  [<user>, i, dived, many, times, for, the, ball...  
3  [my, whole, body, feels, itchy, and, like, its...  
4  [<user>, no, it, is, not, behaving, at, all, i...  
